<a href="https://colab.research.google.com/github/christianlocher/llmrag/blob/main/I_Extra_Vision_LLM_pbp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extra: Komplexe Dokumente interpretieren mit VISION LLM (page by page)

In [ ]:
!pip install pymupdf4llm
import fitz  # Das ist PyMuPDF
import io
import google.generativeai as genai
from google.colab import userdata
from PIL import Image

key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=key)

vision_model = genai.GenerativeModel('gemini-2.5-flash-lite')

pdf_path = "sample_data/13_Basisnormen-für-Tischler-und-Schreiner.pdf"
doc = fitz.open(pdf_path)

chunks = []

print(f"Starte Verarbeitung von {len(doc)} Seiten...")

#pdf_file = genai.upload_file(pdf_path)
#response = vision_model.generate_content([prompt, pdf_file])


for page_num in range(len(doc)):
    page = doc[page_num]

    # Render die Seite als Bild.
    # Matrix(2, 2) verdoppelt die Auflösung, damit die KI auch kleinen Text gut lesen kann.
    zoom_matrix = fitz.Matrix(2, 2)
    pix = page.get_pixmap(matrix=zoom_matrix)
    pix.save(f"sample_data/page-{page.number}.png")  # store image as a PNG as example/debugging
    print(f"Seite {page_num+1} gerendert")

    # Konvertiere das PyMuPDF-Bild in ein PIL-Bild (im Arbeitsspeicher)
    img_data = pix.tobytes("jpeg")
    img = Image.open(io.BytesIO(img_data))
    print(f"Seite {page_num+1} konvertiert")

    # Der Prompt, der Gemini genau sagt, was es mit dem Bild der Seite tun soll
    prompt = """
    Du bist ein Experte für Datenextraktion. Dies ist das Bild einer PDF-Seite.
    Bitte mache Folgendes:
    1. Extrahiere den gesamten sichtbaren Text so exakt wie möglich.
    2. Wenn es Tabellen gibt, wandle diese in sauberes Markdown-Tabellenformat um.
    3. Wenn es Diagramme, Graphen, Logos füge an der
       entsprechenden Stelle im Text eine detaillierte Beschreibung ein (z.B. Trends, Achsen, Kernaussagen).
    4. Wenn es Architektur oder Möbel in den Bildern gibt, so füge an der entsprechenden Stelle eine genaue Beschreibung der dargestellten Elemente ein.
    """

    try:
        # Schicke Prompt und das Bild der Seite an Gemini
        response = vision_model.generate_content([prompt, img])
        print(f"Seite {page_num+1} durch Vision Modell analysiert")
        print(response.text)
        # Speichere das Ergebnis
        chunks.append({
            "page": page_num + 1,
            "page_content": response.text,
            "source": pdf_path
        })

        print(f"Seite {page_num + 1} erfolgreich analysiert.")


    except Exception as e:
        print(f"Fehler bei Seite {page_num + 1}: {e}")


#if chunks:
    #print("\n--- VORSCHAU SEITE 1 ---")
    #print(extracted_pages[0]["content"])
   # print(chunks[0].page_content)
    #print("\n--- VORSCHAU SEITE 2 ---")
    #print(extracted_pages[1]["content"])
  #  print(chunks[1].page_content)
#

#documents = []
#metadatas = []
#ids = []

#for item in extracted_pages:
    # Der Textinhalt, den Gemini erstellt hat
 #   documents.append(item["content"])

    # Metadaten helfen dir später beim Filtern (z.B. nur Seite 5 durchsuchen)
  #  metadatas.append({
   #     "page": item["page"],
    #    "source": pdf_path
   # })

    # Jedes Dokument braucht eine eindeutige ID
  #  ids.append(f"id_page_{item['page']}")

# ==========================================
# 3. In die Collection einspeisen
# ==========================================
#print(f"Lade {len(documents)} Seiten in die ChromaDB hoch...")

#collection.add(
#    documents=documents,
#    metadatas=metadatas,
#    ids=ids
#)

#print("Erfolgreich gespeichert!")